# Lab 10: Explainability with Alibi & SHAP

This notebook uses four distinct explanation objects:

- **Partial-dependence variance (PDV):** a global feature-importance heuristic based on variation in partial-dependence values.
- **Partial-dependence (PD) curves:** average model response as selected features vary over a reference distribution.
- **Image anchors:** locally sufficient superpixel conditions for a prediction from a separate image classifier.
- **SHAP:** local attributions relative to a background dataset; the global chart below aggregates the absolute local attributions.

These methods describe model behavior under assumptions. They do not establish causation or prove that a system is fair or unfair. Complete the TODO code and short-answer cells, then run the notebook from top to bottom.

## Section 0 — Runtime setup

The recommended path is a fresh Google Colab CPU runtime using Python 3.12. Run the next cell once. If Colab asks you to restart the runtime, restart and resume at the version-check cell. The same pins are in `requirements.txt` for a fresh local Python 3.12 environment.

In [ ]:
%pip install -q alibi==0.9.6 click==8.1.8 matplotlib==3.8.4 numpy==1.26.4 pandas==2.2.3 scikit-image==0.22.0 scikit-learn==1.5.2 shap==0.46.0 spacy==3.7.5

In [ ]:
import platform
from importlib.metadata import version
import alibi, matplotlib, numpy as np, pandas as pd, shap, sklearn, skimage, spacy

expected = {
    'alibi': '0.9.6', 'click': '8.1.8', 'matplotlib': '3.8.4',
    'numpy': '1.26.4', 'pandas': '2.2.3', 'scikit-image': '0.22.0',
    'scikit-learn': '1.5.2', 'shap': '0.46.0', 'spacy': '3.7.5',
}
versions = {
    'alibi': alibi.__version__, 'click': version('click'),
    'matplotlib': matplotlib.__version__, 'numpy': np.__version__,
    'pandas': pd.__version__, 'scikit-image': skimage.__version__,
    'scikit-learn': sklearn.__version__, 'shap': shap.__version__,
    'spacy': spacy.__version__,
}
print({'python': platform.python_version(), **versions})
assert tuple(map(int, platform.python_version_tuple()[:2])) == (3, 12), 'Use the supported Python 3.12 runtime.'
assert versions == expected, 'Restart the runtime after installation, then rerun this cell.'

## Section 1 — Tabular data and model setup

We use the historical Boston Housing data only as a compact model-inspection exercise. Several variables encode social and geographic conditions, and the variable named `b` is derived from a racial-demographic measure using an objectionable historical formulation. Do not treat these labels or this model as a valid housing-policy system. That context makes the data useful for forming audit hypotheses, not for declaring fairness from an attribution plot.

In [ ]:
from pathlib import Path

data_path = Path('BostonHousing.csv')
if not data_path.exists():
    try:
        from google.colab import files
        print('Upload BostonHousing.csv from the starter repository.')
        files.upload()
    except ImportError as error:
        raise FileNotFoundError(
            'Place BostonHousing.csv beside this notebook, then rerun the cell.'
        ) from error
if not data_path.exists():
    raise FileNotFoundError('The uploaded file must be named BostonHousing.csv.')
df = pd.read_csv(data_path)
df.head()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

target_name = 'medv'
feature_names = [name for name in df.columns if name != target_name]
X = df[feature_names].to_numpy()
y = df[target_name].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)
predictor = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
predictor.fit(X_train, y_train)
print(f'Train R²: {predictor.score(X_train, y_train):.2f}')
print(f'Test R²: {predictor.score(X_test, y_test):.2f}')

## Section 2 — PDV importance and PD response curves

### 2.1 Partial-dependence variance (PDV)

PDV summarizes how much each feature's partial-dependence values vary. We use it as a **global importance heuristic**: larger variation means the model's average prediction changes more over that feature's evaluated grid. It is not the PD curve, permutation importance, a causal effect, or a guarantee that the feature is useful for every row.

In [ ]:
from io import BytesIO
import matplotlib.pyplot as plt
from IPython.display import Image, display
from alibi.explainers import PartialDependenceVariance, plot_pd_variance

# TODO: create the PDV explainer, compute feature importance, and plot it.
# Hint: call PartialDependenceVariance(predictor=..., feature_names=..., target_names=...)
# and then .explain(X_train, method='importance').
pdv_explainer = ...
pdv_importance = ...
pdv_axes = plot_pd_variance(exp=pdv_importance)
pdv_figure = pdv_axes.ravel()[0].figure
pdv_buffer = BytesIO()
pdv_figure.savefig(pdv_buffer, format='png', bbox_inches='tight')
display(Image(data=pdv_buffer.getvalue()))
plt.close(pdv_figure)

**Q1 — PDV importance: Which features have the largest PDV importance? Explain why this is a global importance heuristic rather than a PD response curve.**

_Your answer:_

### 2.2 One-feature partial dependence

A PD curve shows the model's **average response** as one selected feature varies while predictions are averaged over the other feature values in `X_train`. This is a model-response summary, not a data correlation or a causal effect. When inputs are dependent, replacing one feature across rows can create unrealistic combinations; inspect the observed data range and treat sparse regions cautiously. ICE curves show row-level model responses around that average.

In [ ]:
from alibi.explainers import PartialDependence, plot_pd

pd_explainer = PartialDependence(
    predictor=predictor.predict,
    feature_names=feature_names,
    target_names=[target_name],
)
selected_names = ['rm', 'lstat', 'ptratio']
selected_features = [feature_names.index(name) for name in selected_names]
pd_explanation = pd_explainer.explain(
    X=X_train, features=selected_features, kind='both', grid_resolution=25
)
plot_pd(
    exp=pd_explanation, n_cols=3, n_ice=30, sharey='row', center=False,
    fig_kw={'figheight': 5, 'figwidth': 14},
);

**Q2 — PD response: What average model-response patterns do the curves show? Identify any region where sparse or correlated inputs make extrapolation risky.**

_Your answer:_

### 2.3 Two-feature partial dependence

The heatmap below is still an average model-response surface. It can suggest interactions in the fitted model, but it does not prove that changing either feature would cause the predicted or real outcome to change.

In [ ]:
feature_1, feature_2 = 'rm', 'lstat'  # Try another pair after the first run.
pair = [(feature_names.index(feature_1), feature_names.index(feature_2))]
pair_explanation = pd_explainer.explain(
    X=X_train, features=pair, kind='average', grid_resolution=25
)
plot_pd(
    exp=pair_explanation, n_cols=1, sharey='row', center=False,
    fig_kw={'figheight': 5, 'figwidth': 8},
);

**Q3 — Two-feature PD: Interpret the average-response surface. State one claim it supports about this model and one causal or individual-level claim it does not support.**

_Your answer:_

## Section 3 — Image-anchor exercise

This section trains a small classifier on scikit-learn's bundled handwritten-digit images. An anchor identifies superpixel conditions that are locally sufficient for one image prediction under the explainer's perturbation process. Precision estimates how often the rule preserves the prediction when it applies; coverage estimates how often it applies in sampled perturbations.

This is a separate model and task from the housing regression. Interpret the anchor on its own terms—do **not** compare its superpixels with the tabular PDV or SHAP feature rankings.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier

digits = load_digits()
X_img_train, X_img_test, y_img_train, y_img_test = train_test_split(
    digits.images, digits.target, test_size=0.2, random_state=0, stratify=digits.target
)
image_model = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
image_model.fit(X_img_train.reshape(len(X_img_train), -1), y_img_train)
print(f'Image test accuracy: {image_model.score(X_img_test.reshape(len(X_img_test), -1), y_img_test):.3f}')

In [ ]:
from alibi.explainers import AnchorImage

def image_predict(images):
    return image_model.predict_proba(np.asarray(images).reshape(len(images), -1))

def grid_segments(image):
    # Sixteen stable 2x2 superpixels for an 8x8 digit image.
    return np.repeat(np.repeat(np.arange(16).reshape(4, 4), 2, axis=0), 2, axis=1)

anchor_explainer = AnchorImage(
    image_predict, image_shape=(8, 8, 1), segmentation_fn=grid_segments,
    images_background=X_img_train[..., np.newaxis], seed=0,
)
image_index = 0
image_to_explain = X_img_test[image_index][..., np.newaxis]
predicted_label = image_model.predict(image_to_explain.reshape(1, -1))[0]
anchor = anchor_explainer.explain(
    image_to_explain, threshold=0.85, p_sample=0.5, tau=0.20,
    coverage_samples=2000, max_anchor_size=5,
)
anchor_precision = float(np.asarray(anchor.precision).reshape(-1)[-1])
anchor_coverage = float(np.asarray(anchor.coverage).reshape(-1)[-1])
anchor_segments = list(anchor.raw['feature'])
assert anchor_segments, 'The seeded exercise should produce a non-empty anchor.'
assert anchor.anchor.shape == image_to_explain.shape
print(f'Predicted label: {predicted_label}')
print(f'Retained segments: {anchor_segments}')
print(f'Anchor precision: {anchor_precision:.3f}; coverage: {anchor_coverage:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(image_to_explain[..., 0], cmap='gray')
axes[0].set_title(f'Original (predicted {predicted_label})')
axes[1].imshow(anchor.anchor[..., 0], cmap='gray')
axes[1].set_title('Anchor')
axes[2].imshow(anchor.segments, cmap='tab20')
axes[2].set_title('Superpixels')
for axis in axes: axis.axis('off')
plt.tight_layout()
plt.show()

**Image-anchor reflection: What regions does the anchor retain? Explain the precision/coverage tradeoff and one limitation of treating this local rule as a general explanation of the image model.**

_Your answer:_

## Section 4 — SHAP local attributions and global aggregation

We use a fixed, seeded sample of 100 **training rows** as the SHAP background. `TreeExplainer` uses `feature_perturbation='interventional'`, so it evaluates feature perturbations relative to that empirical background rather than estimating a causal intervention. A different background or feature-dependence treatment can change the attributions.

Each local attribution explains a prediction relative to the background's expected model output. The global chart is `mean(abs(SHAP value))` over the test rows—an aggregation of local absolute attributions, not permutation importance and not PDV.

In [ ]:
rng = np.random.default_rng(0)
background_indices = rng.choice(len(X_train), size=100, replace=False)
shap_background = X_train[background_indices]
shap_explainer = shap.TreeExplainer(
    predictor, data=shap_background, feature_perturbation='interventional',
    feature_names=feature_names,
)
shap_values = shap_explainer(X_test)
assert shap_background.shape == (100, len(feature_names))
assert shap_values.values.shape == X_test.shape
mean_absolute_shap = np.abs(shap_values.values).mean(axis=0)
assert mean_absolute_shap.shape == (len(feature_names),)
print(f'SHAP values shape: {shap_values.values.shape}')
order = np.argsort(mean_absolute_shap)
plt.figure(figsize=(8, 5))
plt.barh(np.asarray(feature_names)[order], mean_absolute_shap[order])
plt.xlabel('Mean absolute local SHAP attribution')
plt.title('Global aggregation of local absolute attributions')
plt.tight_layout()
plt.show()

In [ ]:
local_index = 5
shap.plots.waterfall(shap_values[local_index], max_display=13);

**Q4 — SHAP: Which features have the largest mean absolute local SHAP attributions? Compare this ranking with PDV. Why might these different quantities rank features differently?**

_Your answer:_

## Section 5 — Audit hypothesis

An attribution can show that this fitted model's prediction differs from its background expectation in association with a feature. It cannot by itself show unlawful discrimination, unfair treatment, causal influence, or real-world harm. Use an unexpected or sensitive association to raise an **audit hypothesis**, then name the evidence needed to investigate it.

**Q5 — Audit hypothesis: Identify one feature whose modeled use justifies further fairness or bias auditing. State a concrete follow-up check (for example, subgroup error analysis with appropriately governed demographic data, a data-lineage review, sensitivity testing, or domain review). Why does the attribution alone not settle the question?**

_Your answer:_

## References

1. https://docs.seldon.io/projects/alibi/en/stable/methods/PartialDependence.html
2. https://docs.seldon.io/projects/alibi/en/stable/methods/PartialDependenceVariance.html
3. https://docs.seldon.io/projects/alibi/en/stable/methods/Anchors.html
4. https://shap.readthedocs.io/en/latest/generated/shap.Explainer.html
5. https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html